In [ ]:
# Load the TensorBoard notebook extension.
%load_ext tensorboard

In [ ]:
# Clear any logs from previous runs
!rm -rf ./logs/

In [ ]:
from datetime import datetime
from packaging import version

import tensorflow as tf
from tensorflow import keras

import pandas as pd
tf.debugging.experimental.enable_dump_debug_info('./logs/',
                                                 tensor_debug_mode="FULL_HEALTH", 
                                                 circular_buffer_size=-1)
from keras import backend as K
import numpy as np

print("TensorFlow version: ", tf.__version__)
assert version.parse(tf.__version__).release[0] >= 2, \
    "This notebook requires TensorFlow 2.0 or above."

In [ ]:
load_ferrari_dataset = pd.read_csv('Ferrari_stock.csv')

load_ferrari_dataset.head()

df_dropped_firstcolumn = load_ferrari_dataset.drop(load_ferrari_dataset.columns[0], axis=1)
print(df_dropped_firstcolumn.head())

df_clean = df_dropped_firstcolumn.dropna()


In [ ]:
data_size = len(df_dropped_firstcolumn)

print("Data size: ", data_size)
# 85% of the data is for training.
train_pct = 0.85

train_size = int(data_size * train_pct)

# We want to use as predictors the 'Open', 'High', 'Low' and 'Close' columns, so we will use those as our input data.
x = df_dropped_firstcolumn[['Open', 'High', 'Low', 'Close']]

# Generate the output data.
# y = We want to predict the volume of stocks traded, so we will use the 'Volume' column as our output data.
y = df_dropped_firstcolumn['Volume']

# Split into test and train pairs.
x_train, y_train = x[:train_size], y[:train_size]
x_test, y_test = x[train_size:], y[train_size:]

In [ ]:
from sklearn.preprocessing import MinMaxScaler
logdir = "logs/scalars/" + datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = keras.callbacks.TensorBoard(log_dir=logdir)

# Scale inputs
scaler_x = MinMaxScaler()
x_train_scaled = scaler_x.fit_transform(x_train)
x_test_scaled = scaler_x.transform(x_test)

# Convert y to 2D NumPy arrays
y_train = y_train.to_numpy().reshape(-1, 1)
y_test = y_test.to_numpy().reshape(-1, 1)

# Scale outputs
scaler_y = MinMaxScaler()
y_train_scaled = scaler_y.fit_transform(y_train)
y_test_scaled = scaler_y.transform(y_test)

# Build model
model = keras.models.Sequential([
    keras.layers.Input(shape=(4,)),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(1),
])

# Compile model
model.compile(
    loss='mse',
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    metrics=['mae']
)

# Train
training_history = model.fit(
    x_train_scaled, y_train_scaled,
    validation_data=(x_test_scaled, y_test_scaled),
    epochs=50,
    batch_size=32,
    callbacks=[tensorboard_callback],
)

# Evaluate
loss = model.evaluate(x_test_scaled, y_test_scaled)
print("Test loss:", loss)

In [ ]:
#http://localhost:6006

In [ ]:
%tensorboard --logdir logs/